# ✈️ AeroLex: Evidence-first answers for airline passengers

**Workshop challenge:** build a passenger-rights assistant over EU/US regulations
and Delta contracts of carriage.

> “My flight was cancelled. What refund or compensation applies?”

**Learning goal:** Building a RAG pipeline for a chatbot for airline passengers.

### What is RAG?
RAG stands for Retrieval Augmented Generation. It allows us to provide LLMs with domain-specific information which it may not have seen during training.

### Why retrieve when the model has a huge context window?

**Bring domain evidence beyond the model's training; capacity alone does not select it.**

- **Focus:** retrieve passages relevant to this question, with traceable source IDs.
- **Reuse:** index a corpus once; avoid repeatedly sending every document, hence, reduce token consumption and costs.
- **Needle in the haystack problem** LLMs are biased towards using the information at the beginning or end of the provided context.  
- **Tradeoff:** retrieval adds complexity and can miss evidence. Sources still need refreshing.

A small, bounded document set may be simpler to put directly in the prompt—even with a million-token window available.

### RAG, tools, and agents are building blocks—not rivals

| Need | Useful starting point |
| --- | --- |
| Read a small, bounded document set | Direct / long-context prompt |
| Repeated questions over a large document collection | RAG over a maintained index |
| Live booking status or an action such as rebooking | Authorized API / tool / MCP |
| A multi-step task combining these | Agent using retrieval and tools |

**MCP is a protocol for exposing tools/resources.** An agent can call a RAG search tool through MCP.
Neither MCP nor RAG replace eachother, they are complimentary.

### The pipeline: prepare → retrieve → refine → answer

![Full-pipeline diagram: prepare chunks and indexes; rewrite a question, retrieve with BM25 and vectors, fuse ranks, rerank, and generate with optional profile context.](assets/rag-workshop/pipeline.svg)

- **Build / refresh:** legal PDFs → chunks + IDs → lexical index and document vectors.
- **Per question:** parallel search → RRF → cross-encoder → evidence in the LLM prompt.
- **Optional context:** history clarifies the question; saved profile facts personalize the prompt.

*The following sections build each stage before composing the full pipeline.*

### Google Colab only: prepare the hosted runtime

**Local Conda / venv users:** the next cell does nothing outside Colab.

1. Save a personal notebook copy in Drive; connect a **CPU** runtime (no accelerator required).
2. In Colab's **Secrets** panel, add `GEMINI_API_KEY` and enable notebook access. Never put its value in a cell.
3. Run the next cell **before the application imports**. It clones the public project into `/content`, installs it in this runtime, and loads the secret.

Use the same `COLAB_REF` (branch or commit) as the notebook you opened. For the workshop, use an instructor-tested commit for both.
If installation asks you to restart the session, restart and rerun this cell before continuing.

**First run:** set `CHUNKING_STRATEGY = "hierarchical"`.
Recursive preparation is optional and consumes Gemini quota. New VMs need setup again; saving the notebook does not preserve VM files or packages.

In [1]:
import sys

COLAB_REF = "main" 

if "google.colab" in sys.modules:
    import importlib
    import os
    from pathlib import Path
    import subprocess
    from google.colab import userdata

    colab_repo = Path("/content/aerolex-production-rag")
    colab_url = "https://github.com/nutansahoo/aerolex-production-rag.git"
    if not colab_repo.exists():
        subprocess.run(["git", "clone", "--quiet", colab_url, str(colab_repo)], check=True)
        subprocess.run(["git", "-C", str(colab_repo), "checkout", "--quiet", "--detach", COLAB_REF], check=True)
    else:
        origin = subprocess.run(["git", "-C", str(colab_repo), "remote", "get-url", "origin"], check=True, capture_output=True, text=True).stdout.strip()
        if origin != colab_url:
            raise RuntimeError("Existing /content/aerolex-production-rag is not the expected repository. No files were overwritten.")
        head = subprocess.run(["git", "-C", str(colab_repo), "rev-parse", "HEAD"], check=True, capture_output=True, text=True).stdout.strip()
        requested = subprocess.run(["git", "-C", str(colab_repo), "rev-parse", f"{COLAB_REF}^{{commit}}"], check=True, capture_output=True, text=True).stdout.strip()
        if head != requested:
            raise RuntimeError("Existing checkout does not match COLAB_REF. Save your work and use a fresh runtime or explicitly manage the checkout; no reset was performed.")

    os.chdir(colab_repo)
    subprocess.run([sys.executable, "-m", "pip", "install", "-e", "."], check=True)
    if str(colab_repo) not in sys.path:
        sys.path.insert(0, str(colab_repo))
    importlib.invalidate_caches()
    colab_key = userdata.get("GEMINI_API_KEY")
    if not colab_key:
        raise RuntimeError("Add GEMINI_API_KEY in Colab Secrets and enable notebook access.")
    os.environ["GEMINI_API_KEY"] = colab_key
    del colab_key
    print("Colab setup ready:", sys.version.split()[0], "| revision:", COLAB_REF)
    print("Gemini key configured.")
else:
    print("Local runtime: skipping Colab setup; use your installed Conda or venv environment.")

Local runtime: skipping Colab setup; use your installed Conda or venv environment.


## 0 · Setup

In [2]:
import os
import time
import numpy as np
from pathlib import Path
from aerolex import config
from aerolex.loaders import load_documents
from aerolex.embeddings import GeminiEmbedder
from aerolex.generate import Generator
from aerolex.pipeline import RagPipeline
from aerolex.store import InMemoryVectorStore
from aerolex.chunking import chunk_corpus, load_chunks

In [3]:
# Set API key, LLM, Embedding Model 
EMBED_MODEL = "gemini-embedding-001"
EMBED_DIM = 768
GEN_MODEL = "gemini-3.5-flash-lite"
KEY = config.load_api_key()
CACHE_DIR = config.CACHE_DIR
CHUNKING_STRATEGY = "hierarchical"  # "hierarchical" or "recursive"; restart and rerun after changing
if CHUNKING_STRATEGY not in {"hierarchical", "recursive"}:
    raise ValueError("CHUNKING_STRATEGY must be 'hierarchical' or 'recursive'.")
if not KEY:
    raise RuntimeError("Set GEMINI_API_KEY in the repository-root .env or environment, then restart the kernel.")

## 1 · Hierarchical Chunking

**Preserve legal identity before worrying about chunk size.**

- Detect **Article / § / RULE** boundaries; keep the heading and source on every part.
- Pack paragraphs within each section; use overlapping character windows when a paragraph is too long.
- Retrieve a passage and still identify it as **“EU 261/2004 — Article 7,”** not merely “chunk 42.”

In [4]:
docs = load_documents()
for d in docs:
    print(f"{d.doc_id:16s} {d.doc_type:6s} {len(d.text):>7,d} chars  {d.title}")

# Prefer the pre-built cache (instant); otherwise chunk live.
try:
    hierarchical_chunks = chunk_corpus(docs)
    print(f"\nchunked live: {len(hierarchical_chunks)} hierarchical chunks")

except Exception as exc:
    hierarchical_chunks = load_chunks()
    print(f"\nloaded {len(hierarchical_chunks)} hierarchical chunks from cache")


chunks = hierarchical_chunks
by_id = {c.chunk_id: c for c in chunks}
art7 = next(c for c in chunks if c.section_label == "Article 7")
print("\nExample chunk —", art7.citation)
print(art7.text[:320], "…")

delta_domestic   delta   78,663 chars  Delta Domestic General Rules Tariff (Contract of Carriage)
delta_intl       delta  118,099 chars  Delta International General Rules Tariff (Contract of Carriage)
eu261            eu      27,544 chars  EU Regulation (EC) No 261/2004
cfr260           cfr     26,287 chars  US DOT 14 CFR Part 260

chunked live: 234 hierarchical chunks

Example chunk — EU Regulation (EC) No 261/2004 — Article 7
1.
Where reference is made to this Article, passengers shall
receive compensation amounting to:
(a) EUR 250 for all flights of 1 500 kilometres or less;
(b) EUR 400 for all intra-Community flights of more than
1 500 kilometres, and for all other flights between 1 500
and 3 500 kilometres;
(c) EUR 600 for all flights no …


### Alternative: LangChain recursive character splitting

**Same legal sections and citations; a different body splitter.**

- `RecursiveCharacterTextSplitter` tries paragraphs, lines, spaces, then characters.
- Uses the configured 1,600-character body limit and 150-character target overlap; this is not semantic splitting.
- `alternate_chunks` uses the same `Chunk` schema. The setup cell's `CHUNKING_STRATEGY` selects both chunks and matching vectors downstream.

Prepare recursive caches once with `python -m aerolex.prepare_recursive_cache` (uses Gemini quota).
The cell below loads cached chunks or splits locally without API calls. Restart and rerun top to bottom when switching strategies.

In [5]:
if CHUNKING_STRATEGY == "recursive":
    from aerolex.chunking_recursive import chunk_corpus_recursive

    try:
        alternate_chunks = load_chunks(config.RECURSIVE_CHUNKS_CACHE)
        print(f"loaded {len(alternate_chunks)} recursive chunks from cache")
    except FileNotFoundError:
        alternate_chunks = chunk_corpus_recursive(docs)
        print(f"split locally: {len(alternate_chunks)} recursive chunks (no API calls)")

    alternate_art7 = next(c for c in alternate_chunks if c.section_label == "Article 7")
    print("Alternative example —", alternate_art7.citation)
    print(alternate_art7.text[:320], "…")

    chunks = alternate_chunks if CHUNKING_STRATEGY == "recursive" else hierarchical_chunks
    embeddings_path = config.RECURSIVE_EMBEDDINGS_CACHE
    by_id = {c.chunk_id: c for c in chunks}
    art7 = next(c for c in chunks if c.section_label == "Article 7")
    print(f"Selected: {CHUNKING_STRATEGY} | {len(chunks)} chunks | {embeddings_path.name}")
    # Load the vector store
    store = InMemoryVectorStore.load(embeddings_path)
    chunk_ids = [c.chunk_id for c in chunks]
    matrix = store.matrix

## 2 · Embedding Creation and Caching

**An embedding turns searchable text into a reusable numeric representation.**

![Embedding lifecycle: create and save document vectors at index time; embed a new question and compare it with those vectors at query time.](assets/rag-workshop/embedding-cache.svg)

- **Keep IDs aligned:** every vector must belong to the correct chunk; query/document embeddings need compatible spaces.
- **Rebuild together** when text, chunking, model, or dimensions change.


In [7]:
if not KEY:
    print("GEMINI_API_KEY not set.")

print(f"Embedding {len(chunks)} chunks with {EMBED_MODEL} "
        f"(dim={EMBED_DIM}) …")

t0 = time.time()
embedder = GeminiEmbedder(api_key=KEY, model=EMBED_MODEL, dim=EMBED_DIM)
matrix = embedder.embed_documents([c.searchable_text for c in chunks])
print(f"Embedded in {time.time() - t0:.1f}s, shape={matrix.shape}")

Embedding 234 chunks with gemini-embedding-001 (dim=768) …
Embedded in 179.6s, shape=(234, 768)


In [ ]:
# create vector store
store = InMemoryVectorStore([c.chunk_id for c in chunks], matrix)

# load embeddings (Uncomment and Run only if NOT able to create embeddings)
# store = InMemoryVectorStore.load(Path(CACHE_DIR)/"embeddings.npz")

## 3 · Lexical Search with BM25

**Search by words and identifiers: tokenize → score → rank.**

- **Informative terms matter:** matches on rarer terms usually contribute more.
- **More is not endlessly better:** repeated matches have diminishing gains; document length is accounted for.
- **Token design matters:** keep references such as `260.6` and `261/2004` intact.

BM25 ranks lexical relevance; it is not just a substring or all-words-must-match filter.

### 🛠️ TODO: write `tokenize`

*Use the same token rules as `aerolex.bm25`: lowercase words, the section sign, and numeric references; no stopword filtering. The searches below pass these tokens explicitly.*

In [ ]:
import re

# Keep words, the section sign, and numeric references as single tokens.
_TOKEN = re.compile(r"[a-z]+|\u00A7|\d+(?:[./-]\d+)*")


def tokenize(text: str) -> list[str]:
    # tokens =  #TODO 
    return None

assert tokenize("What is the refund for a cancelled flight?") == ["what", "is", "the", "refund", "for", "a", "cancelled", "flight"]
assert tokenize("See § 260.6 and Regulation 261/2004") == ["see", "§", "260.6", "and", "regulation", "261/2004"]
print("tokenize OK")

In [10]:
from aerolex.bm25 import BM25Index

bm25 = BM25Index([c.chunk_id for c in chunks], [c.searchable_text for c in chunks])
for cid, score in bm25.search(tokenize("§ 260.6 refund for a cancelled flight"), top_k=3):
    print(f"{score:6.2f}  {by_id[cid].citation}")

 16.02  US DOT 14 CFR Part 260 — § 260.6 (part 6/6)
 15.18  US DOT 14 CFR Part 260 — § 260.6 (part 5/6)
 15.00  US DOT 14 CFR Part 260 — § 260.6 (part 3/6)


## 4 · Vector Search and In-Memory Storage

**Search by learned similarity—not only shared words.**

![Toy comparison: BM25 sees literal cat and street in passage A; vector search can recognize the feline and road paraphrase in B. These are illustrative signals, not measured rankings.](assets/rag-workshop/lexical-vs-dense.svg)

- **Dense retrieval:** embed the question; compare its vector direction with document vectors using cosine similarity.
- **Different strengths:** BM25 anchors identifiers; embeddings can connect paraphrases. Neither guarantees relevance.
- **Hybrid retrieval:** run both, then merge their ranked candidates with RRF.

Here, storage is an **in-memory NumPy matrix**, not conversation history or an external vector database.

### 🛠️ TODO: write `cosine_search`

**Normalize → dot product → sort.** A vector should match itself with cosine ≈ 1.
That checks the search mechanics—not whether the embedding model understands every question.

In [ ]:
def cosine_search(query_vec, matrix, ids, top_k=5):

    # 1) L2-normalize query_vec and each row of matrix
    # q_hat = # TODO
    # matrix_hat = #TODO
    #       2) scores = matrix @ query   (cosine, since normalized)
    # scores = # TODO 
    #       3) return the top_k (id, score) pairs, highest first
    top_k = np.argsort(-scores)[:top_k]
    top_chunks = [(ids[i], scores[i]) for i in top_k]
    return top_chunks

# sanity: a row is most similar to itself (cosine ≈ 1.0)
_hit = cosine_search(store.matrix[0], store.matrix, store.ids, top_k=1)[0]
print(_hit[0] == store.ids[0], round(_hit[1], 4))

True 1.0


In [28]:
# Embed a real query (1 API call) and search. Needs a key; else we reuse a chunk vector.
query = "What is the refund for a cancelled flight"

query_vector = GeminiEmbedder(api_key=KEY, model=EMBED_MODEL, dim=EMBED_DIM).embed_query(query)

dense_hits = cosine_search(query_vector, store.matrix, store.ids, top_k=20)
for cid, s in dense_hits:
    print(f"{s:5.3f}  {by_id[cid].citation}")

0.726  Delta International General Rules Tariff (Contract of Carriage) — RULE 23 (part 1/7)
0.724  Delta Domestic General Rules Tariff (Contract of Carriage) — RULE 22 (part 1/6)
0.713  US DOT 14 CFR Part 260 — § 260.6 (part 2/6)
0.698  Delta Domestic General Rules Tariff (Contract of Carriage) — RULE 22 (part 2/6)
0.696  Delta Domestic General Rules Tariff (Contract of Carriage) — RULE 22 (part 5/6)
0.695  US DOT 14 CFR Part 260 — § 260.4 (part 1/3)
0.694  Delta International General Rules Tariff (Contract of Carriage) — RULE 23 (part 5/7)
0.692  Delta International General Rules Tariff (Contract of Carriage) — RULE 23 (part 4/7)
0.689  Delta Domestic General Rules Tariff (Contract of Carriage) — RULE 22 (part 4/6)
0.689  US DOT 14 CFR Part 260 — § 260.1
0.685  US DOT 14 CFR Part 260 — § 260.6 (part 4/6)
0.684  Delta International General Rules Tariff (Contract of Carriage) — RULE 23 (part 6/7)
0.682  US DOT 14 CFR Part 260 — § 260.10
0.682  Delta International General Rules Tariff (C

## 5 · Reciprocal Rank Fusion

**Let each ranked list vote; do not add BM25 scores to cosine scores.**

![RRF example: lexical ranks A B C and vector ranks B C A become rank-based votes; B has the largest combined vote.](assets/rag-workshop/rank-fusion.svg)

- **Rank means position:** first, second, third—not the original score magnitude.
- **A high position earns a bigger vote;** appearing high in multiple lists helps.
- **Missing from a list?** No vote from that list. A candidate can still enter through either retriever.

### Why reciprocal rank? A small, decreasing vote

$$
\mathrm{RRF}(d) = \sum_{L: d \in L} \frac{1}{60 + \mathrm{rank}_L(d)}
$$

| Passage | BM25 rank | Vector rank | Added votes | Total |
| --- | ---: | ---: | --- | ---: |
| B | 2 | 1 | 1/62 + 1/61 | **0.03252** |
| A | 1 | 3 | 1/61 + 1/63 | 0.03227 |
| C | 3 | 2 | 1/63 + 1/62 | 0.03200 |

**Intuition:** reciprocal decay still credits lower ranks; adding `k` softens an outlier's first-place advantage.
This is an established heuristic: the original RRF paper selected **60** in pilot experiments—not a universal optimum.

Ranks here start at **1**; the code uses `rank + 1` because `enumerate` starts at 0.
`k` is smoothing, **not** `top_k` (how many results to keep). These totals are not probabilities.

### 🛠️ TODO: write `rrf`

In [ ]:
def rrf(rankings, k=60, top_k=10):
    scores = {}
    for ranked in rankings:
        for rank, cid in enumerate(ranked):
            chunk_score = scores.get(cid, 0.0)
            # TODO for each chunk compute the RRF Score
            # 
    return sorted(scores.items(), key=lambda kv: kv[1], reverse=True)[:top_k]

assert rrf([["b", "a", "c"], ["b", "a", "d"]])[0][0] == "b"
print("rrf OK")

rrf OK


In [30]:
bm25_ids = [cid for cid, _ in bm25.search(tokenize(query), top_k=20)]
dense_ids = [cid for cid, _ in dense_hits]
fused = rrf([bm25_ids, dense_ids], k=config.RRF_K, top_k=20)
print("Fused top-5:")
for cid, s in fused:
    print(f"{s:.4f}  {by_id[cid].citation}")

Fused top-5:
0.0323  US DOT 14 CFR Part 260 — § 260.6 (part 2/6)
0.0304  US DOT 14 CFR Part 260 — § 260.1
0.0303  Delta International General Rules Tariff (Contract of Carriage) — RULE 23 (part 1/7)
0.0297  US DOT 14 CFR Part 260 — § 260.6 (part 4/6)
0.0296  Delta Domestic General Rules Tariff (Contract of Carriage) — RULE 22 (part 1/6)
0.0292  US DOT 14 CFR Part 260 — § 260.4 (part 1/3)
0.0284  US DOT 14 CFR Part 260 — § 260.6 (part 5/6)
0.0272  Delta International General Rules Tariff (Contract of Carriage) — RULE 23 (part 2/7)
0.0271  US DOT 14 CFR Part 260 — § 260.4 (part 2/3)
0.0265  US DOT 14 CFR Part 260 — § 260.10
0.0159  US DOT 14 CFR Part 260 — § 260.6 (part 1/6)
0.0156  Delta Domestic General Rules Tariff (Contract of Carriage) — RULE 22 (part 2/6)
0.0154  Delta Domestic General Rules Tariff (Contract of Carriage) — RULE 22 (part 5/6)
0.0152  Delta International General Rules Tariff (Contract of Carriage) — RULE 15 (part 4/4)
0.0149  Delta Domestic General Rules Tariff (Cont

## 6 · Cross-Encoder Re-Ranking

**Retrieval finds candidates. A joint reader refines their order.**

![Cross-encoder: each fused candidate is paired with the question, jointly scored, and reordered before selecting LLM context. Toy scores are not probabilities.](assets/rag-workshop/cross-encoder.svg)

- **Unlike cached document embeddings,** each score needs a fresh read of question + passage together.
- **Spend that work on a shortlist,** not the whole corpus; finer relevance judgments can improve ordering.
- **Limits:** scores are not probabilities; reranking cannot recover missed candidates or verify the final answer.

*The cross-encoder runs locally; its public model may download on first use.*

In [31]:
from aerolex.rerank import CrossEncoderReranker

candidates = [(cid, by_id[cid].searchable_text) for cid, _ in fused]
reranked = CrossEncoderReranker().rerank(query, candidates, top_k=20) # config.RERANK_TOP_K)
print("Re-ranked top chunks:")
for cid, s in reranked:
    print(f"{s:7.2f}  {by_id[cid].citation}")
final_chunks = [by_id[cid] for cid, _ in reranked]

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

Re-ranked top chunks:
   6.68  Delta Domestic General Rules Tariff (Contract of Carriage) — RULE 22 (part 1/6)
   6.28  US DOT 14 CFR Part 260 — § 260.6 (part 4/6)
   6.20  US DOT 14 CFR Part 260 — § 260.6 (part 6/6)
   5.77  US DOT 14 CFR Part 260 — § 260.6 (part 1/6)
   5.55  US DOT 14 CFR Part 260 — § 260.6 (part 5/6)
   5.28  US DOT 14 CFR Part 260 — § 260.6 (part 2/6)
   5.24  Delta International General Rules Tariff (Contract of Carriage) — RULE 23 (part 1/7)
   3.78  US DOT 14 CFR Part 260 — § 260.1
   3.31  US DOT 14 CFR Part 260 — § 260.4 (part 1/3)
   2.00  Delta Domestic General Rules Tariff (Contract of Carriage) — RULE 16 (part 4/4)
   1.71  Delta Domestic General Rules Tariff (Contract of Carriage) — RULE 22 (part 5/6)
   1.44  Delta International General Rules Tariff (Contract of Carriage) — RULE 23 (part 5/7)
   0.84  Delta International General Rules Tariff (Contract of Carriage) — RULE 15 (part 4/4)
   0.71  US DOT 14 CFR Part 260 — § 260.10
   0.70  US DOT 14 CFR Par

## 7 · Generate Answer

**Turn supplied evidence into an answer—not an automatic proof.**

- Give the model the question, context passages, and a clear instruction to acknowledge missing evidence.
- Request source tags such as `[[chunk_id]]`; the current generation path returns text **without citation validation**.

*Both the standalone call below and the full `RagPipeline` supply retrieved/reranked passages, rather than the whole corpus.*

In [ ]:
# TODO: Write a system prompt for the AeroLex assistant
system_prompt = (
    
)

In [17]:
user_prompt_template = (
    "{profile_block}\n\n"
    "Context passages:\n{context_text}\n\n"
    "Question: {query}\n\n"
    "Answer (cite every factual claim with [[chunk_id]]):"
)

result = Generator(api_key = KEY, model=GEN_MODEL).generate(
    query, final_chunks,
    system_prompt=system_prompt,
    user_prompt_template=user_prompt_template,
)

print(result)

If a flight is cancelled, the refund provisions depend on the specific rules and regulations applicable:

* **Under US DOT regulations (14 CFR Part 260):** A covered carrier that is the merchant of record must provide a full and prompt refund of the airfare (including taxes and ancillary fees) to a consumer holding a nonrefundable ticket for any cancelled flight if the consumer chooses not to fly on an alternative flight or accept any offered voucher, credit, or other compensation [[cfr260::260-6::1]]. Furthermore, an automatic full refund is due if a flight is canceled and the consumer is not offered an alternative flight, voucher, credit, or other form of compensation [[cfr260::260-6::1]]. 
* **Under Delta Domestic Rules:** If an involuntary refund is required due to a flight cancellation pursuant to Rule 19(A), the refund will be made directly to you as follows:
  * If no portion of the ticket has been used, the refund will equal the fare paid [[delta_domestic::rule-22::0]].
  * If 

## 8 · Conversational Memory

**Make the question self-contained before searching.**

> “What about in the US?” → “US refund rules for cancelled flights?” *(illustrative rewrite)*

History supplies the missing subject; query reformulation carries it into retrieval.
This is conversation context—not the document-vector store.

In [36]:
from aerolex.memory import ConversationMemory

mem = ConversationMemory(api_key=KEY, model=GEN_MODEL)
mem.add_user(query)
mem.add_assistant(result)
print("follow-up : what about in the EU?")
print("standalone:", mem.condense("what about in the EU?"))

follow-up : what about in the EU?
standalone: flight cancellation refund regulations EU


## 9 · Long-Term Persistence

**Remember selected user facts across sessions.**

- SQLite stores facts such as a home airport; the profile can be included in later prompts.
- This demo **explicitly saves** facts—it does not automatically extract every fact from conversation.
- Profile context personalizes an answer.

In [19]:
from aerolex.persistence import ProfileStore

prof = ProfileStore()
prof.remember("demo_user", "home_airport", "JFK")
prof.remember("demo_user", "status", "Delta Platinum Medallion")
print(prof.context_block("demo_user"))

Known facts about the user:
- home_airport: JFK
- status: Delta Platinum Medallion


## 10 · The Full Pipeline

**Connect the stages into one reusable path.**

**Question → rewrite → BM25 + vector → RRF → rerank → context + profile → answer.**


In [37]:
pipe = RagPipeline(
    chunks=chunks,
    store=store,
    bm25=bm25,
    embedder=GeminiEmbedder(api_key=KEY, model=EMBED_MODEL, dim=EMBED_DIM),
    reranker=CrossEncoderReranker(),
    generator=Generator(api_key=KEY, model=GEN_MODEL),
)
out = pipe.answer("If my 4000 km flight is cancelled, what am I owed?",
                    memory=mem, profile=prof, user_id="demo_user",
                    system_prompt=system_prompt,
                    user_prompt_template=user_prompt_template)
print(out["answer"])

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

If your flight is cancelled, you have the right to compensation by the operating air carrier in accordance with Article 7, unless specific exceptions apply regarding the timing of when you were informed [[eu261::article-5::0]]. For a flight of 4,000 kilometres (which falls under flights not falling under EUR 250 or EUR 400 categories), the compensation amount is EUR 600 [[eu261::article-7::0]]. 

Additionally, in the event of a flight cancellation, you shall be offered assistance by the operating air carrier in accordance with Article 8 and Article 9(1)(a) and 9(2) (as well as additional assistance under Article 9 if re-routing departs at least the day after) [[eu261::article-5::0]]. The compensation must be paid in cash, by electronic bank transfer, bank orders, or bank cheques, or with your signed agreement, in travel vouchers and/or other services [[eu261::article-7::0]]. 

The operating air carrier may reduce the compensation by 50% if you are offered re-routing to your final desti

### How different retrieval strategies behave with different queries

Not all combinations of retrieval strategies work well for all queries. Here's an example of a query where vector search doesn't produce the best/most complete answer, but hybrid retrieval does. 

In [39]:
query = "I paid by credit card. How long does the airline have to refund me?"

# Dense retrieval only
query_vector = embedder.embed_query(query)
dense_hits = cosine_search(
    query_vector, store.matrix, store.ids, top_k=20
)

# Use the top 10 dense chunks directly—no BM25, RRF, or reranker
dense_chunks = [by_id[cid] for cid, _ in dense_hits[:10]]

result = Generator(api_key=KEY, model=GEN_MODEL).generate(
    query,
    dense_chunks,
    system_prompt=system_prompt,
    user_prompt_template=user_prompt_template,
)

print(result)

For tickets paid by credit card, Delta will typically issue refunds within seven business days of its initial receipt of the refund request [[delta_intl::rule-23::4], [delta_domestic::rule-22::4]].


In [ ]:
# with cross-encoder reranker
pipe = RagPipeline(
    chunks=chunks,
    store=store,
    bm25=bm25,
    embedder=GeminiEmbedder(api_key=KEY, model=EMBED_MODEL, dim=EMBED_DIM),
    # reranker=CrossEncoderReranker(),
    generator=Generator(api_key=KEY, model=GEN_MODEL),
)
out = pipe.answer("I paid by credit card. How long does the airline have to refund me?",
                    memory=mem, profile=prof, user_id="demo_user",
                    system_prompt=system_prompt,
                    user_prompt_template=user_prompt_template)
print(out["answer"])

When a refund is due pursuant to US DOT regulations, it must be issued promptly in the original form of payment [[cfr260::260-10::0]]. A prompt refund for credit card purchases is defined as being made within 7 business days after the earliest date the refund was requested [[cfr260::260-2::3]]. 

Similarly, under Delta's domestic and international general rules tariffs, tickets paid for by credit card will typically be refunded to the credit card account used to purchase the ticket within seven business days of Delta's initial receipt of the refund request [[delta_intl::rule-23::4], [delta_domestic::rule-22::4]].


Dense answer:

`typically issue the refund within seven business days.`

Hybrid Answer:

`U.S. DOT rules require a refund within seven business days`

The crucial DOT definition chunk:

- Lexical rank: 4
- Dense answer context: absent
- RRF rank: 6

The vector search did not return nonsense. It found a very plausible airline-policy answer. But our question asks what the airline has to do, not what it typically does. Hybrid search can recover evidence that a vector search top-k misses.

#### Here's a query where none of the retrievers worked:

In [43]:
# with cross-encoder reranker
pipe = RagPipeline(
    chunks=chunks,
    store=store,
    bm25=bm25,
    embedder=GeminiEmbedder(api_key=KEY, model=EMBED_MODEL, dim=EMBED_DIM),
    # reranker=CrossEncoderReranker(),
    generator=Generator(api_key=KEY, model=GEN_MODEL),
)
out = pipe.answer("The airline added a connection to my nonstop flight. Can I get my money back instead?",
                    memory=mem, profile=prof, user_id="demo_user",
                    system_prompt=system_prompt,
                    user_prompt_template=user_prompt_template)
print(out["answer"])

Yes, if a change causes you to miss connections or constitutes a significantly delayed or changed flight pursuant to applicable law, Delta will cancel the remaining ticket and refund the unused portion of the ticket and unused ancillary fees (including paid checked bag fees) in the original form of payment upon your request, if Delta does not offer an alternative flight or voucher option, or if you do not respond to the alternative flight or voucher offer prior to the originally scheduled departure time [[delta_domestic::rule-19::0], [delta_intl::rule-20::1]]. Additionally, under US DOT regulations, covered carriers must provide a full and prompt refund of the airfare, including taxes and ancillary fees, for a significantly changed flight if you choose not to fly on the changed flight, accept rebooking, or accept any voucher, credit, or compensation [[cfr260::260-6::1]].


The relevant chunk's wording is a bit different: 

`The consumer is scheduled to travel on an itinerary with more connection points than that of the original itinerary.`

The answer exists but our system is not able to retrieve it. This is where we can try out the following retrieval strategies:
1. Rewrite the query and search with the query + variants:
    - “Nonstop itinerary changed to connecting itinerary”
    - “Increase in connection points significant itinerary change refund”
2. HyDE: Generate a hypothetical explanatory passage, embed it, and retrieve original source passages.
    - `A carrier changes a nonstop itinerary to one with additional connection points, and the passenger wants to reject the itinerary change.`


In [44]:
from google import genai
from aerolex._gemini import generate_text

question = "The airline added a connection to my nonstop flight. Can I get my money back instead?"

prompt = (
    "Write a short hypothetical passenger-rights passage answering this question. "
    "Assume a flight covered by U.S. DOT rules. Use document-style language, "
    "including relevant concepts and conditions, but no citations or statute numbers. "
    "This is a retrieval aid, not verified evidence.\n\n"
    f"Question: {question}"
)
with genai.Client(api_key=KEY) as client:
    hyde_passage = generate_text(client, "gemini-3.5-flash-lite", prompt)

if not hyde_passage.strip():
    raise RuntimeError("Gemini returned an empty HyDE passage.")
print("HYPOTHETICAL PASSAGE:\n", hyde_passage)

hyde_hits = cosine_search(
    embedder.embed_query(hyde_passage),
    store.matrix, store.ids, top_k=10,
)

print("\nRETRIEVED CHUNKS:")
for rank, (cid, score) in enumerate(hyde_hits, 1):
    print(f"{rank:2}. {score:.3f}  {cid}  {by_id[cid].citation}")

print("\nTARGET CHUNK RANKS:")
for target in ("cfr260::260-2::4", "cfr260::260-6::1", "cfr260::260-10::0"):
    rank = next((i for i, (cid, _) in enumerate(hyde_hits, 1) if cid == target), None)
    print(f"{target}: {rank if rank is not None else 'NOT in top 10'}")

HYPOTHETICAL PASSAGE:
 **Passenger Entitlement: Flight Schedule Alteration (Nonstop to Connecting)**

When an operating air carrier alters a confirmed itinerary by introducing an intermediate connection where a nonstop routing was originally purchased, passengers are entitled to a full, original-form-of-payment refund upon request. 

**Conditions and Applicability:**
* **Significance of Change:** The addition of a connection constitutes a significant schedule modification and a fundamental alteration of the transportation contract, rather than a minor time adjustment.
* **Consent:** Passengers are not obligated to accept alternative routing, rerouting, or travel vouchers offered by the carrier.
* **Method of Payment:** Refunds apply to completely unused tickets, as well as the proportional value of partially used itineraries, and must be issued without undue delay upon the passenger's declination of the modified itinerary. 
* **Exclusions:** This right applies regardless of the reason 

In [45]:
hyde_chunks = [by_id[cid] for cid, _ in hyde_hits]

hyde_answer = Generator(api_key=KEY, model=GEN_MODEL).generate(
    question,
    hyde_chunks,
    system_prompt=system_prompt,
    user_prompt_template=user_prompt_template,
)

print(hyde_answer)

Yes. If a flight is significantly changed—such as when an individual with a disability is scheduled to travel through one or more connecting airports that are different from the original itinerary—or if there is a significant change where the consumer chooses not to fly on the significantly delayed or changed flight, accept rebooking, or take any voucher, credit, or other form of compensation, covered carriers that are the merchant of record must provide a full and prompt refund of the airfare, including any taxes and ancillary fees [[cfr260::260-6::1], [cfr260::260-6::3], [cfr260::260-1::0]]. Additionally, under Delta's domestic and international rules (if applicable to your carrier), if an involuntary refund is required due to a significantly changed flight and no portion of the ticket has been used, the refund will be an amount equal to the fare paid [[delta_domestic::rule-22::0], [delta_intl::rule-23::0]].


# Evaluation
- Generated 49 passenger-rights questions covering U.S. regulations, EU rights, Delta policies, and multi-document scenarios, using GPT-6.1 Sol. 
- Compared four retrieval strategies: keyword/BM25, dense/vector, RRF hybrid, and RRF followed by cross-encoder reranking.
- Generated 196 independent answers using gemini-3.5-flash-lite.
- Scored the answers on a 0–3 rubric using LLM-as-a-judge (GPT-6.1 Sol) for correctness and completeness; checked citation validity separately and compared overall and category-level performance.

Rubric Used:
- **3:** Direct, substantially complete, no material substantive error
- **2:** Mostly correct, with a missing qualification or detail
- **1:** Relevant information, but a material omission or misleading condition
- **0:** Wrong regime, no useful answer, or abstention on an answerable question

![Evaluation flowchart comparing lexical, dense, RRF, and RRF with cross-encoder reranking](assets/evaluation_flowchart.svg)

## Strategy comparison

| Strategy | Average / 3 | Mostly or fully correct — score ≥2 | Fully answered — score 3 |
|---|---:|---:|---:|
| **Dense** | **2.31** | **39/49** | 26/49 |
| **RRF** | **2.24** | 35/49 | **29/49** |
| RRF + cross-encoder | 2.02 | 31/49 | 24/49 |
| Lexical/BM25 | 1.78 | 27/49 | 23/49 |